In [103]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

Data Ingestion

In [104]:
df=pd.read_csv(r"C:\Users\hp\OneDrive\Desktop\machine-learning-zoomcamp-homework\week-03-classification\data\course_lead_scoring_2026.csv")

Data Inspection

In [105]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [106]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [107]:
df.describe()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
count,4631.000000,5000.000000,5000.000000,4965.000000,5000.000000
mean,54533.212049,2.089400,4.472600,0.494767,0.576600
std,21245.661710,1.188987,2.435906,0.199431,0.494147
min,12000.000000,0.000000,0.000000,0.010000,0.000000
25%,40458.500000,1.000000,3.000000,0.360000,0.000000
50%,55051.000000,2.000000,4.000000,0.490000,1.000000
75%,71252.500000,3.000000,6.000000,0.630000,1.000000
max,109886.000000,6.000000,13.000000,0.990000,1.000000


Data Preparation

In [108]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [109]:
# Seperating Numerical and Categorical Features
numerical_cols=['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
categorical_cols=['lead_source', 'industry', 'employment_status', 'location']

# Filling Missing Values
df[numerical_cols]=df[numerical_cols].fillna(0.0)
df[categorical_cols]=df[categorical_cols].fillna('NA')

Question - 1 What is the most frequent observation (mode) for the column industry?

In [110]:
print("Most frequent observation for the column 'industry' is : ", df['industry'].mode()[0])

Most frequent observation for the column 'industry' is :  technology


Question - 2 Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.                                                             

What are the two features that have the biggest correlation?

In [111]:
df[numerical_cols].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


Splitting Dataset

In [112]:
df_full_train, df_test=train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val=train_test_split(df_full_train, test_size=0.25, random_state=42)

Separating Inputs and Outputs

In [113]:
X_train=df_train.drop('converted', axis=1)
y_train=df_train['converted']
X_val=df_val.drop('converted', axis=1)
y_val=df_val['converted']
X_test=df_test.drop('converted', axis=1)
y_test=df_test['converted']

In [114]:
print(f"X_train : {X_train.shape}, y_train : {y_train.shape}")
print(f"X_val : {X_val.shape}, y_val : {y_val.shape}")
print(f"X_test : {X_test.shape}, y_test : {y_test.shape}")

X_train : (3000, 8), y_train : (3000,)
X_val : (1000, 8), y_val : (1000,)
X_test : (1000, 8), y_test : (1000,)


Question - 3                                                                                            
1. Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.                                                                        
2. Round the scores to 2 decimals using round(score, 2).                                                               
Which of these variables has the biggest mutual information score?                                                      

In [115]:
from sklearn.metrics import mutual_info_score
print("!! MUTUAL INFORMATION SCORE !!")
for col in categorical_cols:
    print(f"MI for {col} is :", round(mutual_info_score(X_train[col], y_train), 2))

!! MUTUAL INFORMATION SCORE !!
MI for lead_source is : 0.03
MI for industry is : 0.0
MI for employment_status is : 0.02
MI for location is : 0.0


Question - 4                                                                                    
1. Now let's train a logistic regression.
2. Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
3. Fit the model on the training dataset.
4. To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:    
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)               
5. Calculate the accuracy on the validation dataset and round it to 2 decimal digits.                                   
What accuracy did you get?

In [116]:
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score
transformation=ColumnTransformer(transformers=[
                ('encode', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_cols),
                ('scaling', StandardScaler(), numerical_cols)
])
X_train_transformed=transformation.fit_transform(X_train)
X_val_transformed=transformation.transform(X_val)
X_test_transformed=transformation.transform(X_test)
X_train_final=pd.DataFrame(X_train_transformed, columns=transformation.get_feature_names_out())
X_val_final=pd.DataFrame(X_val_transformed, columns=transformation.get_feature_names_out())
X_test_final=pd.DataFrame(X_test_transformed, columns=transformation.get_feature_names_out())
model=LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_final, y_train)
y_pred=model.predict(X_val_final)
print("Accuracy Score is : ", round(accuracy_score(y_val, y_pred), 2))

Accuracy Score is :  0.73


Question - 5                                                                                                
1. Let's find the least useful feature using the feature elimination technique.                             
2. Train a model using the same features and parameters as in Q4 (without rounding).                                        
3. Now exclude each feature from this set and train a model without it. Record the accuracy for each model.                 
4. For each feature, calculate the difference between the original accuracy and the accuracy without the feature.           
Which of following feature has the smallest difference?

In [120]:
parent_feature={'lead_source': [col for col in X_train_final.columns if 'lead_source' in col],
                'number_of_courses_viewed': ['scaling__number_of_courses_viewed'],
                'interaction_count': ['scaling__interaction_count']}
print("Original Accuracy Score is '0.73'.")
for feature_name, col_to_drop in parent_feature.items():
    model=LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_final.drop(col_to_drop, axis=1), y_train)
    y_pred=model.predict(X_val_final.drop(col_to_drop, axis=1))
    accuracy=accuracy_score(y_val, y_pred)
    print(f"Accuracy score after removing '{feature_name}' column is : {accuracy}")
    print("Difference b/w original accuracy and accuracy after removing feature : ", round(0.73-accuracy, 4))

Original Accuracy Score is '0.73'.
Accuracy score after removing 'lead_source' column is : 0.722
Difference b/w original accuracy and accuracy after removing feature :  0.008
Accuracy score after removing 'number_of_courses_viewed' column is : 0.717
Difference b/w original accuracy and accuracy after removing feature :  0.013
Accuracy score after removing 'interaction_count' column is : 0.736
Difference b/w original accuracy and accuracy after removing feature :  -0.006


Question - 6                                                                                                    
1. Now let's train a regularized logistic regression.                                                               
2. Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].                           
3. Train models using all the features as in Q4.                                                                
4. Calculate the accuracy on the validation dataset and round it to 3 decimal digits.                                       
Which of these C leads to the best accuracy on the validation set?                      

In [121]:
C=[0.000001, 0.00001, 0.0001, 0.001]
for val in C:
    model=LogisticRegression(solver='liblinear', C=val, max_iter=1000, random_state=42)
    model.fit(X_train_final, y_train)
    y_pred=model.predict(X_val_final)
    print(f"For C={val}, the Accuracy Score is : {accuracy_score(y_val, y_pred)}")

For C=1e-06, the Accuracy Score is : 0.712
For C=1e-05, the Accuracy Score is : 0.712
For C=0.0001, the Accuracy Score is : 0.713
For C=0.001, the Accuracy Score is : 0.717
